# Error taxonomy, backoff, and idempotent tools

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 11 §11.4 · type: walkthrough*

**One-line promise:** build retry logic that retries only what's *retryable*, backs off with jitter, honors `retry-after`, and survives an ambiguous timeout without firing a side effect twice — all driven offline by a fake transport.

## 🧠 Why this matters

A model API is an external dependency with all the classical failure modes of one, plus a few of its own. It fails in **categories**, and the retryable column is the whole game: retrying a `400` wastes quota on a request that can never succeed, while failing hard on a `429` turns a momentary blip into an outage. The genuinely scary entry is the **timeout** — the outcome is *unknown*, so a blind retry of a tool-executing turn can send the email or refund the order twice.

We drive every branch with a **fake transport** that raises `429` / `500` / timeout on a schedule, so you can watch real retry logic work without a key, a network, or a cent of spend.

## Objectives & prereqs

**By the end you can:**
- Classify an API error as retryable vs terminal from the taxonomy table.
- Implement `with_retries(...)`: exponential backoff **with jitter**, an attempt cap, and `retry-after` honored.
- Explain why a timeout is different, and make a side-effecting tool **idempotent** so a replay is harmless.
- Configure the SDK's own `max_retries`/timeouts instead of stacking blind loops.

**Prereqs:** notebook **11-01** (response shape & stop reason). No API key needed — the demo is fully offline.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv from requirements.txt). No network is used.
import os
import time
import random
import itertools
from dataclasses import dataclass

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# This notebook is offline by design: the fake transport exercises every retry
# branch for free. MOCK=0 is optional and NOT required to learn the lesson.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(11)  # makes the jittered delays below reproducible

# Speed knob: real backoff sleeps seconds; we scale it down so the notebook is snappy.
TIME_SCALE = 0.01  # set to 1.0 to feel real backoff durations
print(f"MOCK mode: {MOCK}  | fake transport drives the retry path offline")

## 1 · The error taxonomy (§11.4)

The classical HTTP taxonomy applies, with LLM-specific twists. Encode it as data so the retry logic can ask one question: *is this error in the retryable set?*

In [ ]:
# A faithful stand-in for the SDK's exception hierarchy. The real anthropic SDK
# raises anthropic.RateLimitError / .InternalServerError / .APITimeoutError, etc.
class APIError(Exception):
    status = None

class BadRequestError(APIError):      status = 400   # malformed: bad schema / overflow
class AuthenticationError(APIError):  status = 401   # auth / permission
class PermissionError_(APIError):     status = 403
class NotFoundError(APIError):        status = 404   # unknown model / endpoint
class RequestTooLargeError(APIError): status = 413   # shrink the input
class RateLimitError(APIError):       status = 429   # RPM and/or TPM exhausted
class InternalServerError(APIError):  status = 500   # provider-side error
class OverloadedError(APIError):      status = 529   # provider overloaded (also 503)
class APITimeoutError(APIError):      status = None  # no response — OUTCOME UNKNOWN

# The retryable set — the only column that matters for the retry decision.
RETRYABLE = (RateLimitError, InternalServerError, OverloadedError, APITimeoutError)

TAXONOMY = [
    (400, "Malformed request: bad schema, context overflow", "No — fix the bug"),
    (401, "Auth or permission problem",                      "No — fix config"),
    (403, "Permission problem",                              "No — fix config"),
    (404, "Unknown model or endpoint",                       "No — fix config"),
    (413, "Request too large",                               "No — shrink input"),
    (429, "Rate limited: RPM or TPM exhausted",              "Yes, with backoff"),
    (500, "Provider-side error",                             "Yes, with backoff"),
    (529, "Provider overloaded (also 503)",                  "Yes, with backoff"),
    (None, "Timeout — outcome UNKNOWN",                      "Carefully — see §4"),
]
for status, meaning, retry in TAXONOMY:
    print(f"{str(status):>4}  {meaning:<45}  {retry}")

**What you just saw.** Don't-retry errors (`400/401/403/404/413`) are *your* bug — a retry only wastes quota. Retry-with-backoff errors (`429/500/529/503`) are transient provider conditions. The timeout sits apart: the request may have *completed* on the server even though you saw nothing.

## 2 · A fake transport that fails on a schedule

To exercise the retry path deterministically we need a `call()` that fails predictably. This transport raises a scripted sequence of errors, then returns a success — no network, no key, no cost.

In [ ]:
@dataclass
class FakeResponse:
    text: str
    headers: dict


class FakeTransport:
    """Raises a scripted error sequence, then succeeds. Records every attempt."""

    def __init__(self, script):
        # script: list of exception instances (or None for 'succeed here')
        self._script = iter(script)
        self.attempts = 0

    def call(self):
        self.attempts += 1
        step = next(self._script, None)  # past the script => success
        if step is not None:
            raise step
        return FakeResponse(text="OK: model replied", headers={})


# A 429 carrying a retry-after header (providers tell you how long to wait).
rate_limited = RateLimitError("429 Too Many Requests")
rate_limited.headers = {"retry-after": "2"}  # seconds

demo = FakeTransport([rate_limited, InternalServerError("500"), None])
print("transport ready: will raise 429, then 500, then succeed")

## 3 · `with_retries`: backoff **with jitter**, capped, honoring `retry-after`

This is the workhorse from §11.4. Two non-negotiables: **jitter** (so a fleet of clients doesn't retry in lockstep and re-spike the provider) and an **attempt cap** (so a persistent failure surfaces instead of looping forever). If the error carries `retry-after`, that wins over the computed backoff.

In [ ]:
def _retry_after(err):
    """Return the server-requested wait in seconds, if present."""
    headers = getattr(err, "headers", None) or {}
    val = headers.get("retry-after")
    return float(val) if val is not None else None


def with_retries(call, *, max_attempts=5, base=1.0, log=None):
    """Retry `call` on RETRYABLE errors with exponential backoff + jitter.
    Honors retry-after. Re-raises terminal errors immediately and gives up
    after max_attempts."""
    for attempt in range(max_attempts):
        try:
            return call()
        except RETRYABLE as err:
            if attempt == max_attempts - 1:
                raise  # out of budget — let it surface
            server_wait = _retry_after(err)
            backoff = base * (2 ** attempt) + random.uniform(0, 1)  # full jitter
            delay = server_wait if server_wait is not None else backoff
            if log is not None:
                log.append((attempt, type(err).__name__, round(delay, 2),
                            "retry-after" if server_wait is not None else "backoff"))
            time.sleep(delay * TIME_SCALE)
        # NOTE: terminal errors (400/401/403/404/413) are NOT caught here —
        # they propagate immediately, exactly as intended.


log = []
result = with_retries(demo.call, log=log)
print("result      :", result.text)
print("attempts    :", demo.attempts)
for a, kind, delay, src in log:
    print(f"  retry #{a}: saw {kind:<20} waited {delay:>5}s via {src}")

**What you just saw.** The 429's `retry-after: 2` was honored verbatim; the 500 used computed backoff with jitter; the third attempt succeeded. Three attempts, one success — and a terminal error would have skipped all of this and surfaced at once.

Now confirm the *other* half of the contract — a terminal error must **not** be retried:

In [ ]:
bad = FakeTransport([BadRequestError("400 bad input_schema")])
try:
    with_retries(bad.call)
except BadRequestError as e:
    print("propagated immediately (no retry):", e)
print("attempts made:", bad.attempts, "  <- exactly 1; we did NOT waste quota")

> ⚠️ **Pitfall — the retry storm.** A loop with *no jitter, no cap, and retries on 4xx* is a self-inflicted outage: every client backs off to the same instant and stampedes the recovering provider, while 400s burn quota forever. Jitter de-synchronizes the fleet; the cap turns an outage into a clean error you can alert on; the retryable filter keeps you from hammering on bugs.

## 4 · 🔮 Predict: a tool-executing turn times out — then you retry

Scenario: the model's turn calls a `send_refund` tool. The HTTP call **times out** — you got no response. Your retry logic sees `APITimeoutError` (retryable) and calls again.

Before running the next cell, predict: **how many times does the refund actually fire?** Why is a timeout categorically different from a 429 here?

In [ ]:
# A side-effecting 'tool' with NO idempotency. Each successful call refunds money.
refund_ledger = []

def send_refund_naive(order_id, amount):
    refund_ledger.append((order_id, amount))   # the real-world side effect
    return {"refunded": amount, "order_id": order_id}


class FlakyTool:
    """Simulates: the side effect FIRES, but the response is lost to a timeout."""
    def __init__(self, fail_times):
        self._left = fail_times
    def call(self):
        # The danger: the side effect happens BEFORE the connection drops.
        send_refund_naive("ORD-9912", 49.99)
        if self._left > 0:
            self._left -= 1
            raise APITimeoutError("read timeout — response never arrived")
        return FakeResponse(text="refund confirmed", headers={})


flaky = FlakyTool(fail_times=2)  # times out twice (but refunds each time!), then 'succeeds'
with_retries(flaky.call)
print("refunds fired:", len(refund_ledger), "->", refund_ledger)

**What you just saw.** The refund fired **three times** for one order. A timeout means *outcome unknown* — the server may have completed the work you never saw — so a blind retry duplicates the side effect. A 429, by contrast, is safe to retry because the request never executed.

### The fix: idempotency keys (the Ch 29 pattern, applied to tools)

Attach a stable **idempotency key** to each logical operation. The downstream system records the key on first success and *ignores* replays carrying the same key. Now a replayed turn is harmless — exactly what makes retrying through an ambiguous timeout safe.

In [ ]:
# Downstream 'refund service' that dedupes on an idempotency key.
_processed_keys = {}

def send_refund_idempotent(order_id, amount, *, idempotency_key):
    if idempotency_key in _processed_keys:
        return _processed_keys[idempotency_key]  # replay => same result, no new effect
    receipt = {"refunded": amount, "order_id": order_id, "key": idempotency_key}
    _processed_keys[idempotency_key] = receipt
    return receipt


# One key per logical operation — derive it from the turn/tool_use id, NOT per attempt.
key = "refund:ORD-9912:tooluse_01"
real_ledger = []

class IdempotentTool:
    def __init__(self, fail_times):
        self._left = fail_times
    def call(self):
        before = key in _processed_keys
        receipt = send_refund_idempotent("ORD-9912", 49.99, idempotency_key=key)
        if not before and key in _processed_keys:
            real_ledger.append(receipt)  # only count the FIRST real effect
        if self._left > 0:
            self._left -= 1
            raise APITimeoutError("timeout again")
        return receipt

with_retries(IdempotentTool(fail_times=2).call)
print("actual refunds applied:", len(real_ledger), "->", real_ledger)
assert len(real_ledger) == 1, "idempotency must collapse replays to one effect"
print("replays were absorbed — the order was refunded exactly once.")

**What you just saw.** Same two timeouts, same retries — but the refund applied **once**. The downstream service deduped on the key; the extra attempts returned the cached receipt with no new effect.

## 5 · Don't stack blind loops — configure the SDK

The real SDKs already retry transparently a couple of times and let you set timeouts. Prefer configuring them over wrapping yet another loop on top (which would multiply attempts: your 5 × the SDK's 2 = 10).

In [ ]:
# Illustrative configuration — runs only on the live path so the notebook stays offline.
if not MOCK and os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    client = anthropic.Anthropic(max_retries=2, timeout=30.0)  # configure, don't stack
    print("configured SDK:", client)
else:
    print("MOCK mode: showing the config you'd set, without instantiating a live client:")
    print('    anthropic.Anthropic(max_retries=2, timeout=30.0)')
    print("Rule of thumb: let the SDK own a small retry budget; reserve with_retries for")
    print("cross-call concerns (idempotency, circuit breaking) — don't double up loops.")

## 🎯 Senior lens

Backoff alone cannot survive *sustained* overload — requests pile up behind retries until everything is slow and still failing. Production adds the other half: a client-side **concurrency cap** (a semaphore around the LLM client), a **shed/defer queue** for low-priority work (Ch 31), and a **circuit breaker** that fails fast when the provider is down instead of burning your latency budget rediscovering it on every request (Ch 26). Retries handle the blip; the breaker and the semaphore handle the storm. All three belong inside the one client — which you build next.

## Recap

- The API fails in **categories**; encode the taxonomy and retry *only* the retryable set (`429/500/529/503/timeout`).
- `with_retries`: exponential backoff **with jitter**, an attempt **cap**, and `retry-after` honored over computed backoff.
- Terminal errors (`400/401/403/404/413`) must propagate immediately — retrying them only wastes quota.
- A **timeout is outcome-unknown**; blindly retrying a tool turn duplicates side effects.
- **Idempotency keys** make a replayed turn harmless — the prerequisite for safely retrying through a timeout.
- Configure the SDK's own `max_retries`/`timeout`; don't stack blind loops on top.

## Exercises

Predict the result before running each.

1. **Exhaust the budget.** Build a `FakeTransport` that raises `InternalServerError` more times than `max_attempts`. What exception surfaces, and after how many attempts? Add a final `429` and confirm it changes nothing.
2. **Cap the backoff.** Add a `max_delay` ceiling to `with_retries` so a high attempt count can't sleep for minutes. Predict attempt #6's delay with and without the cap.
3. **Respect TPM vs RPM.** Give a 429 a `retry-after` of `0.5`s and confirm it's honored over the computed backoff. Why might a provider return a *shorter* wait than your exponential formula?
4. **Key derivation.** Change the idempotency key to include a per-attempt counter and re-run section 4's fix. Predict the refund count, then explain why the key must be tied to the *logical operation*, not the *attempt*.

In [ ]:
# Exercise 1 — your code here

In [ ]:
# Exercise 2 — your code here

In [ ]:
# Exercise 3 — your code here

In [ ]:
# Exercise 4 — your code here

## Next

- ⬅️ **Previous:** [`11-01-sdk-shapes-and-the-response.ipynb`](./11-01-sdk-shapes-and-the-response.ipynb).
- ➡️ **Next notebook:** [`11-03-caching-batching-and-the-llm-client.ipynb`](./11-03-caching-batching-and-the-llm-client.ipynb) — cut the two biggest costs and fold retries, the semaphore, and the circuit breaker into the single-door `LLMClient`.
- 📘 **Blueprint:** the production retry/timeout/circuit-breaker wrapper lives in [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/).
- 🏗️ **Capstone:** resilience graduates into the platform's `llm/` package (checkpoint `ch11-llm-client`).
- See the book **§11.4** for the taxonomy table and backoff code; idempotency returns in **Ch 29**, full resilience in **Ch 26**.